In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import os
from google.colab import drive

# 1. Google Driveのマウント
drive.mount('/content/drive')

# 2. パスの設定
INPUT_ROOT = Path('/content/drive/MyDrive/石河研/spectrum/tsukuba')
OUTPUT_DIR = Path('/content/drive/MyDrive/石河研/spectrum')

# 【修正1】Google Drive上であることの確認
if not str(INPUT_ROOT).startswith('/content/drive/MyDrive/'):
    raise ValueError(f"【処理停止】INPUT_ROOTが /content/drive/MyDrive/ 配下にありません: {INPUT_ROOT}")
if not str(OUTPUT_DIR).startswith('/content/drive/MyDrive/'):
    raise ValueError(f"【処理停止】OUTPUT_DIRが /content/drive/MyDrive/ 配下にありません: {OUTPUT_DIR}")

# 【修正2】チェックポイント関連パスを最初から定義
CHECKPOINT_ROOT = OUTPUT_DIR / "ape_HSR_r12_350_1050_checkpoints"
RUN_MANIFEST_PATH = CHECKPOINT_ROOT / "run_manifest.json"
RESUME_STATE_PATH = CHECKPOINT_ROOT / "resume_state.json"
PROCESSING_LOCK_PATH = CHECKPOINT_ROOT / "processing_lock.json"
BATCH_DIR = CHECKPOINT_ROOT / "batches"
ERROR_DIR = CHECKPOINT_ROOT / "errors"
TEMP_DIR = CHECKPOINT_ROOT / "temporary"

FINAL_PICKLE_PATH = OUTPUT_DIR / "target_ape_tsukuba_HSR_r12_350_1050.pkl"
FINAL_JSON_PATH = OUTPUT_DIR / "target_ape_tsukuba_HSR_r12_350_1050_quality_report.json"

# 【修正3】既存成果物の確認
paths_to_check = {
    "FINAL_PICKLE_PATH": FINAL_PICKLE_PATH,
    "FINAL_JSON_PATH": FINAL_JSON_PATH,
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "RUN_MANIFEST_PATH": RUN_MANIFEST_PATH,
    "RESUME_STATE_PATH": RESUME_STATE_PATH,
    "PROCESSING_LOCK_PATH": PROCESSING_LOCK_PATH,
    "BATCH_DIR": BATCH_DIR,
    "ERROR_DIR": ERROR_DIR,
    "TEMP_DIR": TEMP_DIR,
}

print("--- パスの存在確認 ---")
path_status = {}
for name, p in paths_to_check.items():
    exists = p.exists()
    path_status[name] = exists
    print(f"{name}: {'存在する' if exists else '存在しない'} ({p})")

if path_status["FINAL_PICKLE_PATH"] or path_status["FINAL_JSON_PATH"]:
    raise FileExistsError("【処理停止】最終成果物（pickleまたはJSON）が既に存在します。上書き防止のため停止します。")

# 【修正4】新規／再開候補の判定
if not path_status["CHECKPOINT_ROOT"]:
    mode = "新規"
    print("\n実行候補モード：新規")
else:
    mode = "再開"
    print("\n実行候補モード：再開\n既存チェックポイントの検証が必要です")

if path_status["PROCESSING_LOCK_PATH"]:
    print("処理ロックあり：次の検証セルでheartbeatを確認する必要があります")

# バッチファイルのカウント
batch_parquet_count = len(list(BATCH_DIR.glob("batch_*.parquet"))) if path_status["BATCH_DIR"] else 0
batch_json_count = len(list(BATCH_DIR.glob("batch_*_quality.json"))) if path_status["BATCH_DIR"] else 0

# 【修正5】表示内容
print("\n--- 確認結果サマリー ---")
print(f"INPUT_ROOT: {INPUT_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"CHECKPOINT_ROOT: {CHECKPOINT_ROOT}")
print(f"最終pickleの存在: {path_status['FINAL_PICKLE_PATH']}")
print(f"最終品質JSONの存在: {path_status['FINAL_JSON_PATH']}")
print(f"run_manifestの存在: {path_status['RUN_MANIFEST_PATH']}")
print(f"resume_stateの存在: {path_status['RESUME_STATE_PATH']}")
print(f"processing_lockの存在: {path_status['PROCESSING_LOCK_PATH']}")
print(f"発見した既存batch parquet数: {batch_parquet_count}")
print(f"発見した既存batch quality JSON数: {batch_json_count}")
print(f"実行候補モード: {mode}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- パスの存在確認 ---
FINAL_PICKLE_PATH: 存在しない (/content/drive/MyDrive/石河研/spectrum/target_ape_tsukuba_HSR_r12_350_1050.pkl)
FINAL_JSON_PATH: 存在しない (/content/drive/MyDrive/石河研/spectrum/target_ape_tsukuba_HSR_r12_350_1050_quality_report.json)
CHECKPOINT_ROOT: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints)
RUN_MANIFEST_PATH: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/run_manifest.json)
RESUME_STATE_PATH: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/resume_state.json)
PROCESSING_LOCK_PATH: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/processing_lock.json)
BATCH_DIR: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/batches)
ERROR_DIR: 存在しない (/content/drive/MyDrive/石河研/spectrum/ape_HSR_r12_350_1050_checkpoints/errors)
TE

In [3]:
import re
from datetime import datetime
from collections import defaultdict

print("--- HSRファイルの検索開始 ---")
print(f"検索ルート: {INPUT_ROOT}")

# 正規表現パターン
pattern_hsr = re.compile(r"^10HSR\d{6}_\d{3}\.csv$")
pattern_tsr = re.compile(r"^10TSR.*\.csv$")
pattern_met = re.compile(r"^10MET.*\.csv$")

all_csv_count = 0
hsr_files = []
excluded_tsr_count = 0
excluded_met_count = 0
other_excluded_count = 0

# 【修正1・2】ファイル検索 (is_fileとfullmatch)
for path in INPUT_ROOT.rglob("*.csv"):
    if not path.is_file():
        continue

    all_csv_count += 1
    filename = path.name

    if pattern_hsr.fullmatch(filename):
        hsr_files.append(path)
    elif pattern_tsr.match(filename):
        excluded_tsr_count += 1
    elif pattern_met.match(filename):
        excluded_met_count += 1
    else:
        other_excluded_count += 1

# 【修正3】HSRファイルのソート (相対パスで決定論的に)
hsr_files = sorted(hsr_files, key=lambda p: p.relative_to(INPUT_ROOT).as_posix())
hsr_count = len(hsr_files)

# 【修正4】相対パス一覧の保持
hsr_relative_paths = [
    p.relative_to(INPUT_ROOT).as_posix()
    for p in hsr_files
]

# 【修正5】同一ファイル名の重複確認
name_to_paths = defaultdict(list)
for rel_p in hsr_relative_paths:
    filename = rel_p.split('/')[-1]
    name_to_paths[filename].append(rel_p)

duplicate_names = {name: paths for name, paths in name_to_paths.items() if len(paths) > 1}
if duplicate_names:
    print("\n【処理停止】同一ファイル名が複数の場所に存在します。重複行の原因となるため停止します。")
    for name, paths in duplicate_names.items():
        print(f"重複ファイル名: {name}")
        for p in paths:
            print(f"  - 相対パス: {p}")
    print(f"重複ファイル名数: {len(duplicate_names)}")
    raise ValueError("重複ファイル名が検出されました。")

# 【修正6】見込み日付の抽出とエラーハンドリング
dates = []
invalid_date_files = []

for p, rel_p in zip(hsr_files, hsr_relative_paths):
    # 10HSRyymmdd_ppp.csv -> yymmdd部分を抽出
    date_str = p.name[5:11]
    try:
        # 2000年代と仮定
        date_obj = datetime.strptime("20" + date_str, "%Y%m%d").date()
        dates.append(date_obj)
    except ValueError:
        invalid_date_files.append((p.name, rel_p, date_str))

if invalid_date_files:
    print("\n【処理停止】日付変換に失敗したファイルがあります。")
    for name, rel_p, d_str in invalid_date_files:
        print(f"ファイル名: {name}")
        print(f"相対パス: {rel_p}")
        print(f"抽出した日付文字列: {d_str}")
    raise ValueError("日付変換に失敗したファイルが存在します。")

# 【修正7】件数の一致確認
if len(dates) != hsr_count:
    raise RuntimeError(f"HSRファイル数({hsr_count})と日付抽出成功数({len(dates)})が一致しません。")

min_date = min(dates) if dates else None
max_date = max(dates) if dates else None

print("\n--- 検索結果 ---")
print(f"発見した全CSV数: {all_csv_count}")
print(f"HSRとして採用するファイル数: {hsr_count}")
print(f"除外されたTSR数: {excluded_tsr_count}")
print(f"除外されたMET数: {excluded_met_count}")
print(f"その他除外数 (出力ファイルや条件不一致): {other_excluded_count}")

if hsr_count == 0:
    raise ValueError("【処理停止】対象となるHSRファイルが0件です。")

# 【修正8】先頭・末尾5件の相対パス表示
print("\n--- 採用ファイル先頭5件 (相対パス) ---")
for rp in hsr_relative_paths[:5]:
    print(rp)

print("\n--- 採用ファイル末尾5件 (相対パス) ---")
for rp in hsr_relative_paths[-5:]:
    print(rp)

print("\n--- 見込み日付範囲 (ファイル名より) ---")
print(f"最小日付: {min_date}")
print(f"最大日付: {max_date}")

# 【修正9】追加の確認事項表示
print("\n--- 品質・整合性チェック結果 ---")
print(f"重複ファイル名数: {len(duplicate_names)}")
print(f"日付変換失敗数: {len(invalid_date_files)}")
print(f"hsr_relative_pathsの件数: {len(hsr_relative_paths)}")
print(f"HSRファイル数と日付抽出成功数の一致: {len(dates) == hsr_count}")


--- HSRファイルの検索開始 ---
検索ルート: /content/drive/MyDrive/石河研/spectrum/tsukuba

--- 検索結果 ---
発見した全CSV数: 3680
HSRとして採用するファイル数: 1824
除外されたTSR数: 1796
除外されたMET数: 60
その他除外数 (出力ファイルや条件不一致): 0

--- 採用ファイル先頭5件 (相対パス) ---
201101/10HSR110101_301.csv
201101/10HSR110102_301.csv
201101/10HSR110103_301.csv
201101/10HSR110104_301.csv
201101/10HSR110105_301.csv

--- 採用ファイル末尾5件 (相対パス) ---
201512/10HSR151227_301.csv
201512/10HSR151228_301.csv
201512/10HSR151229_301.csv
201512/10HSR151230_301.csv
201512/10HSR151231_301.csv

--- 見込み日付範囲 (ファイル名より) ---
最小日付: 2011-01-01
最大日付: 2015-12-31

--- 品質・整合性チェック結果 ---
重複ファイル名数: 0
日付変換失敗数: 0
hsr_relative_pathsの件数: 1824
HSRファイル数と日付抽出成功数の一致: True


In [4]:
import json
import hashlib
import math
from datetime import date, timedelta, datetime, timezone
import collections

print("--- マニフェスト事前検証 (読み取り専用) ---")

# 【1. HSRの日付完全性確認】
start_date = date(2011, 1, 1)
end_date = date(2015, 12, 31)
total_calendar_days = (end_date - start_date).days + 1

date_to_files = collections.defaultdict(list)
for p, rel_p in zip(hsr_files, hsr_relative_paths):
    date_str = p.name[5:11]
    d = datetime.strptime("20" + date_str, "%Y%m%d").date()
    date_to_files[d].append(rel_p)

present_days = len(date_to_files)
missing_dates = []
for i in range(total_calendar_days):
    d = start_date + timedelta(days=i)
    if d not in date_to_files:
        missing_dates.append(d.isoformat())

multiple_files_dates = {k.isoformat(): v for k, v in date_to_files.items() if len(v) > 1}

# 【2. 地点番号の確認】
site_counts = collections.defaultdict(int)
invalid_sites = []
for p, rel_p in zip(hsr_files, hsr_relative_paths):
    site = p.name[-7:-4] # 例: 10HSR151231_301.csv -> 301
    site_counts[site] += 1
    if site != "301":
        invalid_sites.append(rel_p)

if invalid_sites:
    print("\n【処理停止】地点番号301以外のファイルが検出されました。")
    for inv_p in invalid_sites:
        print(f"  - {inv_p}")
    raise ValueError("不正な地点番号が存在します。")

# 【3. CONFIGとconfig_hash】
CONFIG = {
    "pipeline_version": "1.0.0",
    "dataset_name": "experiment_B_HSR",
    "plane": "HSR",
    "w_min": 350,
    "w_max": 1050,
    "remarks_used": [1, 2],
    "duplicate_policy": "error",
    "ape_valid_range_eV": [1.2, 2.2],
    "source_timezone": "Asia/Tokyo",
    "datetime_storage": "timezone-naive local time",
    "source_file_pattern": "^10HSR\\d{6}_\\d{3}\\.csv$",
    "formula": "1239.84193 * sum(G) / sum(G * wavelength)",
    "batch_size": 25,
    "input_root": str(INPUT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "expected_site_numbers": [301]
}

config_json_str = json.dumps(
    CONFIG,
    sort_keys=True,
    ensure_ascii=False,
    separators=(",", ":")
)
config_hash = hashlib.sha256(config_json_str.encode('utf-8')).hexdigest()

# 【4. ファイルメタデータ】
file_metadata = {}
for p, rel_p in zip(hsr_files, hsr_relative_paths):
    st = p.stat()
    file_metadata[rel_p] = {
        "relative_path": rel_p,
        "filename": p.name,
        "size": st.st_size,
        "mtime_ns": st.st_mtime_ns
    }
if len(file_metadata) != hsr_count:
    raise RuntimeError("ファイルメタデータ件数がHSRファイル数と一致しません。")

# 【5. 決定論的バッチ構成】
total_files = len(hsr_relative_paths)
batch_size = CONFIG["batch_size"]
total_batches = math.ceil(total_files / batch_size)

batches_config = {}
assigned_files = set()
duplicates_in_batches = 0

for i in range(total_batches):
    b_id = f"batch_{i:05d}"
    start_idx = i * batch_size
    end_idx = min(start_idx + batch_size, total_files)
    batch_files = hsr_relative_paths[start_idx:end_idx]
    batches_config[b_id] = batch_files

    for f in batch_files:
        if f in assigned_files:
            duplicates_in_batches += 1
        assigned_files.add(f)

unassigned_count = total_files - len(assigned_files)
if len(assigned_files) != total_files or duplicates_in_batches > 0 or unassigned_count > 0:
    raise RuntimeError("バッチ構成に重複または未割り当てが存在します。")

first_batch_id = list(batches_config.keys())[0]
last_batch_id = list(batches_config.keys())[-1]
first_batch_size = len(batches_config[first_batch_id])
last_batch_size = len(batches_config[last_batch_id])

# 【6. manifest_preview】
manifest_preview = {
    **CONFIG,
    "config_hash": config_hash,
    "total_hsr_files": total_files,
    "total_batches": total_batches,
    "sorted_hsr_files": hsr_relative_paths,
    "file_metadata": file_metadata,
    "batches": batches_config,
    "missing_dates": missing_dates,
    "station_file_counts": dict(site_counts),
    "manifest_created_at": datetime.now(timezone.utc).isoformat()
}

# 【7. 実行直前状態の再確認】
if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists():
    raise FileExistsError("【処理停止】最終成果物が既に存在します。")

if mode == "新規" and CHECKPOINT_ROOT.exists():
    raise FileExistsError("【処理停止】modeは新規ですが、CHECKPOINT_ROOTが既に存在します。別セッションの干渉の可能性があります。")

if mode == "再開":
    if not RUN_MANIFEST_PATH.exists():
        raise FileNotFoundError("【処理停止】再開モードですが run_manifest.json が見つかりません。")

    with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
        existing_manifest = json.load(f)

    if existing_manifest.get("config_hash") != config_hash:
        raise ValueError("【処理停止】抽出条件や設定内容 (config_hash) が前回と異なります。")
    if existing_manifest.get("total_hsr_files") != total_files:
        raise ValueError("【処理停止】ファイル数が一致しません。")
    if existing_manifest.get("total_batches") != total_batches:
        raise ValueError("【処理停止】バッチ数が一致しません。")
    if existing_manifest.get("sorted_hsr_files") != hsr_relative_paths:
        raise ValueError("【処理停止】HSRファイルの順序が一致しません。")
    if existing_manifest.get("batches") != batches_config:
        raise ValueError("【処理停止】バッチ構成が一致しません。")

    old_metadata = existing_manifest.get("file_metadata", {})
    for rel_p, meta in file_metadata.items():
        if rel_p not in old_metadata:
            raise ValueError(f"【処理停止】新しいファイルが見つかりました: {rel_p}")
        old_meta = old_metadata[rel_p]
        if old_meta["size"] != meta["size"] or old_meta["mtime_ns"] != meta["mtime_ns"]:
            raise ValueError(f"【処理停止】ファイルサイズまたは更新日時(mtime_ns)が変化しています: {rel_p}")

# 【8. 最終表示】
print("\n--- 事前検証結果 ---")
print(f"実行候補モード: {mode}")
print(f"config_hash: {config_hash}")
print(f"地点番号別ファイル数: {dict(site_counts)}")
print(f"総暦日数: {total_calendar_days} 日")
print(f"HSR存在日数: {present_days} 日")
print(f"HSR欠測日数: {len(missing_dates)} 日")
if missing_dates:
    print(f"欠測日一覧: {missing_dates}")
if multiple_files_dates:
    print(f"複数ファイルが存在する日: {multiple_files_dates}")
print(f"総ファイル数: {total_files}")
print(f"総バッチ数: {total_batches}")
print(f"最初のバッチ ({first_batch_id}) サイズ: {first_batch_size}")
print(f"最後のバッチ ({last_batch_id}) サイズ: {last_batch_size}")
print(f"ファイルメタデータ件数: {len(file_metadata)}")
print(f"重複割り当て数: {duplicates_in_batches}")
print(f"未割り当て数: {unassigned_count}")
print("\n✅ マニフェスト事前検証：成功")
print("※このセルではGoogle Driveへの書き込み（ファイル・ディレクトリ作成等）は一切行っていません。")


--- マニフェスト事前検証 (読み取り専用) ---

--- 事前検証結果 ---
実行候補モード: 新規
config_hash: f483833250e9533f86829d95aa5e6f86b4b99853bae16deff1b49e74dceeccd9
地点番号別ファイル数: {'301': 1824}
総暦日数: 1826 日
HSR存在日数: 1824 日
HSR欠測日数: 2 日
欠測日一覧: ['2011-04-29', '2011-04-30']
総ファイル数: 1824
総バッチ数: 73
最初のバッチ (batch_00000) サイズ: 25
最後のバッチ (batch_00072) サイズ: 24
ファイルメタデータ件数: 1824
重複割り当て数: 0
未割り当て数: 0

✅ マニフェスト事前検証：成功
※このセルではGoogle Driveへの書き込み（ファイル・ディレクトリ作成等）は一切行っていません。


In [6]:
import json
import uuid
import os
from datetime import datetime, timezone

print("--- 新規モード限定：安全な初期化とロック取得 ---")

# 【1. 新規モード限定】
if mode != "新規":
    print("再開モードでは専用のロック検証・取得セルを使用してください。")
    raise RuntimeError("このセルは新規モード専用です。")

# 【2. 実行直前の再確認】
if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists() or CHECKPOINT_ROOT.exists() or \
   RUN_MANIFEST_PATH.exists() or RESUME_STATE_PATH.exists() or PROCESSING_LOCK_PATH.exists():
    raise FileExistsError("【処理停止】最終成果物またはチェックポイントが既に存在します。mode判定後に状態が変化した可能性があります。")

# 【4. 孤立したステージングの検出】
staging_pattern = ".ape_HSR_r12_350_1050_initializing_*"
existing_stagings = list(OUTPUT_DIR.glob(staging_pattern))
if existing_stagings:
    print("\n【処理停止】過去の初期化中断と思われるステージングディレクトリが存在します。")
    for s_dir in existing_stagings:
        print(f"ディレクトリ名: {s_dir.name}")
        st = s_dir.stat()
        mtime = datetime.fromtimestamp(st.st_mtime).isoformat()
        print(f"更新日時: {mtime}")
        files = [f.name for f in s_dir.iterdir()]
        print(f"内部ファイル一覧: {files}")
    raise FileExistsError("孤立したステージングディレクトリが検出されました。安全のため自動削除・上書きは行わず停止します。")

# セッションと日時の生成
session_id = str(uuid.uuid4())
current_time = datetime.now(timezone.utc)

# 【3. ステージングディレクトリ】
STAGING_ROOT = OUTPUT_DIR / f".ape_HSR_r12_350_1050_initializing_{session_id}"
STAGING_ROOT.mkdir(parents=True, exist_ok=False)

STAGING_BATCH_DIR = STAGING_ROOT / "batches"
STAGING_ERROR_DIR = STAGING_ROOT / "errors"
STAGING_TEMP_DIR = STAGING_ROOT / "temporary"

STAGING_BATCH_DIR.mkdir(exist_ok=False)
STAGING_ERROR_DIR.mkdir(exist_ok=False)
STAGING_TEMP_DIR.mkdir(exist_ok=False)

staging_manifest_path = STAGING_ROOT / "run_manifest.json"
staging_resume_path = STAGING_ROOT / "resume_state.json"
staging_lock_path = STAGING_ROOT / "processing_lock.json"

# 【5. JSONの安全な保存関数】
def atomic_write_json(data_dict, target_path, session_id):
    temp_name = f"{target_path.stem}_{session_id}.tmp.json"
    temp_path = target_path.parent / temp_name

    # 1. 保存先と同じディレクトリに一時JSONを作る
    with open(temp_path, "x", encoding="utf-8") as f:
        # 2, 3, 4. json.dump, flush, fsync
        json.dump(data_dict, f, indent=4, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())

    # 5. 一時JSONを読み直す
    with open(temp_path, "r", encoding="utf-8") as f:
        read_back = json.load(f)

    # 6. 元のPython辞書と一致確認
    if read_back != data_dict:
        raise ValueError(f"【処理停止】一時JSONの内容が元の辞書と一致しません: {temp_path}")

    # 7. JSONとして再度読み込めることを確認
    try:
        with open(temp_path, "r", encoding="utf-8") as f:
            json.load(f)
    except Exception as e:
        raise ValueError(f"【処理停止】一時JSONのパースに失敗しました: {e}")

    # 8. os.replaceで正式なJSON名へ変更
    os.replace(temp_path, target_path)

    # 9. 正式JSONを再度読み直して一致確認
    with open(target_path, "r", encoding="utf-8") as f:
        final_read = json.load(f)
    if final_read != data_dict:
        raise ValueError(f"【処理停止】正式JSONの内容が元の辞書と一致しません: {target_path}")

# 【6. ステージングへ保存する内容】
initial_resume_state = {
    "pipeline_version": CONFIG["pipeline_version"],
    "config_hash": config_hash,
    "status": "initialized",
    "completed_batch_ids": [],
    "completed_source_files": [],
    "pending_batch_ids": list(batches_config.keys()),
    "failed_batch_ids": [],
    "total_batches": total_batches,
    "completed_batches": 0,
    "completed_files": 0,
    "total_files": total_files,
    "last_completed_batch": None,
    "last_completed_source_file": None,
    "last_update": current_time.isoformat(),
    "current_session_id": session_id
}

lock_data = {
    "session_id": session_id,
    "config_hash": config_hash,
    "start_time": current_time.isoformat(),
    "heartbeat": current_time.isoformat(),
    "status": "running",
    "lock_schema_version": "1.0"
}

# 保存実行
atomic_write_json(manifest_preview, staging_manifest_path, session_id)
atomic_write_json(initial_resume_state, staging_resume_path, session_id)
atomic_write_json(lock_data, staging_lock_path, session_id)

# 【7. ステージング全体の検証】
if not staging_manifest_path.exists() or not staging_resume_path.exists() or not staging_lock_path.exists():
    raise RuntimeError(f"【処理停止】ステージング内に必要なJSONファイルが揃っていません。\nSTAGING_ROOT: {STAGING_ROOT}")

with open(staging_manifest_path, "r", encoding="utf-8") as f:
    v_manifest = json.load(f)
with open(staging_resume_path, "r", encoding="utf-8") as f:
    v_resume = json.load(f)
with open(staging_lock_path, "r", encoding="utf-8") as f:
    v_lock = json.load(f)

if v_manifest["config_hash"] != config_hash:
    raise ValueError("manifestのconfig_hashが現在値と一致しません。")
if v_resume["config_hash"] != config_hash:
    raise ValueError("resume_stateのconfig_hashが現在値と一致しません。")
if v_lock["config_hash"] != config_hash:
    raise ValueError("lockのconfig_hashが現在値と一致しません。")
if v_resume["current_session_id"] != session_id or v_lock["session_id"] != session_id:
    raise ValueError("session_idが一致しません。")
if v_manifest["total_hsr_files"] != 1824:
    raise ValueError("manifestの総ファイル数が1,824ではありません。")
if v_manifest["total_batches"] != 73:
    raise ValueError("manifestの総バッチ数が73ではありません。")
if len(v_resume["pending_batch_ids"]) != 73:
    raise ValueError("pending_batch_idsが73件ではありません。")
if len(v_resume["completed_batch_ids"]) != 0:
    raise ValueError("completed_batch_idsが空ではありません。")
if len(v_resume["completed_source_files"]) != 0:
    raise ValueError("completed_source_filesが空ではありません。")

if not STAGING_BATCH_DIR.exists() or not STAGING_ERROR_DIR.exists() or not STAGING_TEMP_DIR.exists():
    raise RuntimeError("ステージング内の必須ディレクトリが存在しません。")
if any(STAGING_BATCH_DIR.iterdir()):
    raise RuntimeError("batchesディレクトリが空ではありません。")
if any(STAGING_ERROR_DIR.iterdir()):
    raise RuntimeError("errorsディレクトリが空ではありません。")

# temporaryディレクトリ及びSTAGING_ROOT直下に一時ファイルが残っていないか
if any(STAGING_ROOT.glob("*.tmp.json")):
    raise RuntimeError("保存先ディレクトリにatomic_write_jsonの一時ファイルが残っています。")
if any(STAGING_TEMP_DIR.iterdir()):
    raise RuntimeError("temporaryディレクトリが空ではありません。")

# 【8. 正式名称への切り替え直前確認】
if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists() or CHECKPOINT_ROOT.exists():
    raise FileExistsError("【処理停止】正式名称への切り替え直前に競合ファイルまたはディレクトリが検出されました。")

# 【9. 正式CHECKPOINT_ROOTへの切り替え】
# ディレクトリのrenameによるアトミックな作成
try:
    STAGING_ROOT.rename(CHECKPOINT_ROOT)
except Exception as e:
    raise RuntimeError(f"【処理停止】正式CHECKPOINT_ROOTへのrenameに失敗しました: {e}")

# 【10. 正式化後の再検証】
if not CHECKPOINT_ROOT.exists():
    raise RuntimeError("CHECKPOINT_ROOTが存在しません。renameに失敗した可能性があります。")
if STAGING_ROOT.exists():
    raise RuntimeError("元のSTAGING_ROOTが存在しています。renameが正常に完了していません。")

with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as f:
    f_manifest = json.load(f)
with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    f_resume = json.load(f)
with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    f_lock = json.load(f)

if f_manifest != v_manifest or f_resume != v_resume or f_lock != v_lock:
    raise RuntimeError("正式化後にJSONの内容が変化しています。")

if f_lock["session_id"] != session_id:
    raise ValueError("正式化後のsession_idが一致しません。")
if f_lock["config_hash"] != config_hash:
    raise ValueError("正式化後のconfig_hashが一致しません。")
if f_lock["status"] != "running":
    raise ValueError("正式化後のstatusが正しくありません。")

# 【12. 最終表示】
print("\n--- 初期化完了 ---")
print(f"session_id: {session_id}")
print(f"config_hash: {config_hash}")
print(f"正式CHECKPOINT_ROOT: {CHECKPOINT_ROOT}")
print("manifest検証結果: 成功")
print("resume_state検証結果: 成功")
print("processing_lock検証結果: 成功")
print(f"総ファイル数: {f_manifest['total_hsr_files']}")
print(f"総バッチ数: {f_manifest['total_batches']}")
print(f"pending batch数: {len(f_resume['pending_batch_ids'])}")
print(f"completed batch数: {len(f_resume['completed_batch_ids'])}")
print(f"初期化ステータス: {f_resume['status']}")
print("\n✅ 初期化とロック取得が完全に成功しました。次にCSV処理へ進めます。")


--- 新規モード限定：安全な初期化とロック取得 ---


FileExistsError: 【処理停止】最終成果物またはチェックポイントが既に存在します。mode判定後に状態が変化した可能性があります。

In [7]:
import json
import hashlib
import os
from datetime import datetime, timezone

print("--- 既存チェックポイント読み取り専用監査 ---")

# 【1. 最終成果物の確認】
print("\n【1. 最終成果物の確認】")
for name, path in [("FINAL_PICKLE_PATH", FINAL_PICKLE_PATH), ("FINAL_JSON_PATH", FINAL_JSON_PATH)]:
    if path.exists():
        st = path.stat()
        mtime = datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat()
        print(f"{name}: 存在する - {path} (Size: {st.st_size}, Mtime: {mtime})")
    else:
        print(f"{name}: 存在しない")

# 【2. CHECKPOINT_ROOTの全内容】
print("\n【2. CHECKPOINT_ROOTの全内容】")
expected_items = {
    "run_manifest.json", "resume_state.json", "processing_lock.json",
    "batches", "errors", "temporary"
}
unexpected_items = []
if CHECKPOINT_ROOT.exists():
    for p in CHECKPOINT_ROOT.rglob("*"):
        rel_p = p.relative_to(CHECKPOINT_ROOT).as_posix()
        st = p.stat()
        mtime = datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat()
        ftype = "Dir " if p.is_dir() else "File"
        print(f"- {rel_p} ({ftype}, Size: {st.st_size}, Mtime: {mtime})")

        # 期待するファイル構造からの逸脱を記録
        if p.parent == CHECKPOINT_ROOT and p.name not in expected_items:
            unexpected_items.append(rel_p)
        elif p.parent != CHECKPOINT_ROOT:
            # 今回の段階ではサブディレクトリ配下には一切ファイルが存在しないことが期待される
            unexpected_items.append(rel_p)
else:
    print("CHECKPOINT_ROOTが存在しません。")

# 【3. ディレクトリ内部の確認】
print("\n【3. ディレクトリ内部の確認】")
batch_parquet_count = len(list(BATCH_DIR.glob("batch_*.parquet"))) if BATCH_DIR.exists() else 0
batch_json_count = len(list(BATCH_DIR.glob("batch_*_quality.json"))) if BATCH_DIR.exists() else 0
batches_files = len(list(BATCH_DIR.iterdir())) if BATCH_DIR.exists() else 0
errors_files = len(list(ERROR_DIR.iterdir())) if ERROR_DIR.exists() else 0
temp_files = len(list(TEMP_DIR.iterdir())) if TEMP_DIR.exists() else 0
tmp_files = len(list(CHECKPOINT_ROOT.rglob("*.tmp"))) + len(list(CHECKPOINT_ROOT.rglob("*.tmp.json")))
staging_dirs = len(list(OUTPUT_DIR.glob(".ape_HSR_r12_350_1050_initializing_*")))

print(f"batches内の総ファイル数: {batches_files}")
print(f"batch parquet数: {batch_parquet_count}")
print(f"batch quality JSON数: {batch_json_count}")
print(f"errors内の総ファイル数: {errors_files}")
print(f"temporary内の総ファイル数: {temp_files}")
print(f"一時ファイル (*.tmp, *.tmp.json) 数: {tmp_files}")
print(f"initialization用ステージングディレクトリ数: {staging_dirs}")

# 【4. 3つのJSONの読み取り】
print("\n【4. 3つのJSONの読み取り】")
json_data = {}
json_status = {}

for name, path in [("run_manifest", RUN_MANIFEST_PATH), ("resume_state", RESUME_STATE_PATH), ("processing_lock", PROCESSING_LOCK_PATH)]:
    status = {"exists": path.exists(), "valid_json": False, "is_dict": False, "keys": [], "size": 0, "sha256": "", "error": None}
    if status["exists"]:
        try:
            st = path.stat()
            status["size"] = st.st_size
            with open(path, "rb") as f:
                raw_bytes = f.read()
                status["sha256"] = hashlib.sha256(raw_bytes).hexdigest()

            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
                status["valid_json"] = True
                if isinstance(data, dict):
                    status["is_dict"] = True
                    status["keys"] = list(data.keys())
                    json_data[name] = data
        except Exception as e:
            status["error"] = f"{type(e).__name__}: {e}"
    json_status[name] = status
    print(f"- {name}.json:")
    print(f"  存在: {status['exists']}, 正常読込: {status['valid_json']}, dict型: {status['is_dict']}")
    print(f"  サイズ: {status['size']} bytes, SHA-256: {status['sha256']}")
    if status["is_dict"]:
        print(f"  キー: {status['keys']}")
    if status["error"]:
        print(f"  エラー: {status['error']}")

# 【5. run_manifestの検証】
print("\n【5. run_manifestの検証】")
manifest_valid = False
manifest_diff_keys = []
if "run_manifest" in json_data:
    m = json_data["run_manifest"]
    v_results = {
        "config_hash_match_current": m.get("config_hash") == config_hash,
        "config_hash_match_preview": m.get("config_hash") == manifest_preview.get("config_hash"),
        "total_hsr_files": m.get("total_hsr_files") == 1824,
        "total_batches": m.get("total_batches") == 73,
        "sorted_hsr_files": m.get("sorted_hsr_files") == hsr_relative_paths,
        "file_metadata": m.get("file_metadata") == manifest_preview.get("file_metadata"),
        "batches": m.get("batches") == batches_config,
        "missing_dates": m.get("missing_dates") == ["2011-04-29", "2011-04-30"],
        "expected_site_numbers": m.get("expected_site_numbers") == [301],
        "station_file_counts": m.get("station_file_counts") == {"301": 1824},
        "pipeline_version": m.get("pipeline_version") == "1.0.0"
    }
    for k, v in v_results.items():
        print(f"  {k}: {v}")

    manifest_valid = all(v_results.values())

    for k in set(m.keys()).union(set(manifest_preview.keys())):
        if m.get(k) != manifest_preview.get(k):
            manifest_diff_keys.append(k)
    print(f"  manifest_previewとの相違キー: {manifest_diff_keys}")
else:
    print("  検証スキップ (データなし)")

# 【6. resume_stateの検証】
print("\n【6. resume_stateの検証】")
resume_valid = False
if "resume_state" in json_data:
    r = json_data["resume_state"]
    v_results = {
        "config_hash_match": r.get("config_hash") == config_hash,
        "status": r.get("status") == "initialized",
        "total_batches": r.get("total_batches") == 73,
        "total_files": r.get("total_files") == 1824,
        "completed_batches": r.get("completed_batches") == 0,
        "completed_files": r.get("completed_files") == 0,
        "completed_batch_ids_empty": r.get("completed_batch_ids") == [],
        "completed_source_files_empty": r.get("completed_source_files") == [],
        "failed_batch_ids_empty": r.get("failed_batch_ids") == [],
        "pending_batch_ids_count": len(r.get("pending_batch_ids", [])) == 73,
        "pending_batch_ids_match": r.get("pending_batch_ids") == list(batches_config.keys()),
        "last_completed_batch": r.get("last_completed_batch") is None,
        "last_completed_source_file": r.get("last_completed_source_file") is None,
        "current_session_id_not_empty": bool(r.get("current_session_id"))
    }
    for k, v in v_results.items():
        print(f"  {k}: {v}")
    resume_valid = all(v_results.values())
else:
    print("  検証スキップ (データなし)")

# 【7. processing_lockの検証】
print("\n【7. processing_lockの検証】")
lock_missing_keys = []
heartbeat_elapsed_mins = -1.0
lock_session_match = False

if "processing_lock" in json_data:
    l = json_data["processing_lock"]
    print(f"  session_id: {l.get('session_id')}")
    print(f"  start_time: {l.get('start_time')}")
    print(f"  heartbeat: {l.get('heartbeat')}")
    print(f"  status: {l.get('status')}")

    has_config_hash = "config_hash" in l
    has_schema_version = "lock_schema_version" in l
    print(f"  config_hashが存在するか: {has_config_hash}")
    print(f"  lock_schema_versionが存在するか: {has_schema_version}")

    if not has_config_hash: lock_missing_keys.append("config_hash")
    if not has_schema_version: lock_missing_keys.append("lock_schema_version")
    if lock_missing_keys:
        print("  ※移行が必要: 一部キーが不足しています。")

    rs_session_id = json_data.get("resume_state", {}).get("current_session_id")
    lock_session_match = (rs_session_id == l.get("session_id")) and bool(rs_session_id)
    print(f"  resume_state.current_session_idとlock.session_idが一致するか: {lock_session_match}")

    try:
        st = datetime.fromisoformat(l.get("start_time", ""))
        hb = datetime.fromisoformat(l.get("heartbeat", ""))
        tz_aware = (st.tzinfo is not None) and (hb.tzinfo is not None)
        print(f"  start_timeとheartbeatがtimezone-awareか: {tz_aware}")

        now_utc = datetime.now(timezone.utc)
        if hb.tzinfo is None:
            hb = hb.replace(tzinfo=timezone.utc)
        heartbeat_elapsed_mins = (now_utc - hb).total_seconds() / 60.0
        print(f"  heartbeatから現在UTC時刻までの経過分数: {heartbeat_elapsed_mins:.2f} 分")
    except Exception as e:
        print(f"  日時のパースエラー: {e}")
else:
    print("  検証スキップ (データなし)")

# 【8. 監査判定】
print("\n【8. 監査判定】")
decision = "RESET_REQUIRES_EXPLICIT_CONFIRMATION"
reason = []

if FINAL_PICKLE_PATH.exists() or FINAL_JSON_PATH.exists():
    reason.append("最終成果物が存在します")
if not manifest_valid:
    reason.append("manifestが不完全または不一致")
if not resume_valid:
    reason.append("resume_stateが不完全または不一致")
if not lock_session_match:
    reason.append("session_id不一致または存在しない")
if batches_files > 0 or errors_files > 0 or temp_files > 0:
    reason.append("バッチ/エラー/一時ディレクトリ内にファイルが存在します")
if tmp_files > 0:
    reason.append("一時ファイル(*.tmp)が存在します")
if unexpected_items:
    reason.append("予期せぬファイル/ディレクトリが存在します")

if not reason:
    decision = "ADOPTABLE_WITH_MIGRATION"
else:
    decision = "RESET_REQUIRES_EXPLICIT_CONFIRMATION"

print(f"判定結果: {decision}")
if reason:
    print("理由: " + ", ".join(reason))

# 【9. 最終表示】
print("\n--- 監査結果サマリー ---")
print(f"最終pickleの存在: {FINAL_PICKLE_PATH.exists()}")
print(f"最終品質JSONの存在: {FINAL_JSON_PATH.exists()}")
print("JSON 3件の読み込み結果:")
for k, v in json_status.items():
    print(f"  {k}: 読込={'成功' if v['valid_json'] else '失敗'}")
print(f"manifest検証結果: {'成功' if manifest_valid else '失敗'}")
print(f"resume_state検証結果: {'成功' if resume_valid else '失敗'}")
print(f"lock検証結果: セッション一致={lock_session_match}")
print(f"lockに不足するキー: {lock_missing_keys}")
print(f"heartbeat経過分数: {heartbeat_elapsed_mins:.2f} 分")
print(f"batch parquet数: {batch_parquet_count}")
print(f"batch quality JSON数: {batch_json_count}")
print(f"temporaryファイル数: {temp_files}")
print(f"unexpected_items: {unexpected_items}")
print(f"\n総合監査判定: {decision}")
print("※このセルではGoogle Driveへの書き込みを一切行っていません。")

--- 既存チェックポイント読み取り専用監査 ---

【1. 最終成果物の確認】
FINAL_PICKLE_PATH: 存在しない
FINAL_JSON_PATH: 存在しない

【2. CHECKPOINT_ROOTの全内容】
- batches (Dir , Size: 4096, Mtime: 2026-07-30T15:09:27+00:00)
- errors (Dir , Size: 4096, Mtime: 2026-07-30T15:09:27+00:00)
- temporary (Dir , Size: 4096, Mtime: 2026-07-30T15:09:27+00:00)
- run_manifest.json (File, Size: 567394, Mtime: 2026-07-30T15:09:27+00:00)
- resume_state.json (File, Size: 2255, Mtime: 2026-07-30T15:09:27+00:00)
- processing_lock.json (File, Size: 192, Mtime: 2026-07-30T15:09:27+00:00)

【3. ディレクトリ内部の確認】
batches内の総ファイル数: 0
batch parquet数: 0
batch quality JSON数: 0
errors内の総ファイル数: 0
temporary内の総ファイル数: 0
一時ファイル (*.tmp, *.tmp.json) 数: 0
initialization用ステージングディレクトリ数: 0

【4. 3つのJSONの読み取り】
- run_manifest.json:
  存在: True, 正常読込: True, dict型: True
  サイズ: 567394 bytes, SHA-256: 9d4c7fb99a481789e530c6318cbeb91082d51bd92af9d42e27c095aa05fe7fda
  キー: ['pipeline_version', 'dataset_name', 'plane', 'w_min', 'w_max', 'remarks_used', 'duplicate_policy', 'ape_valid_ra

In [10]:
import json
import hashlib
import os
from datetime import datetime, timezone

print("--- processing_lock.json の安全なスキーマ移行 ---")

# 【1. 現在のランタイムがロック所有者であることの確認】
print("\n【1. ランタイムのロック所有権確認】")
runtime_session_id = globals().get("session_id")
if not runtime_session_id:
    raise RuntimeError("【処理停止】メモリ上に session_id が存在しません。現在のランタイムがロック所有者であることを証明できません。")

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    old_lock = json.load(f)

with open(RESUME_STATE_PATH, "r", encoding="utf-8") as f:
    current_resume = json.load(f)

if old_lock.get("status") != "running":
    raise ValueError(f"【処理停止】既存ロックの status が 'running' ではありません: {old_lock.get('status')}")
if not old_lock.get("session_id"):
    raise ValueError("【処理停止】既存ロックに session_id が存在しません。")
if old_lock.get("session_id") != current_resume.get("current_session_id"):
    raise ValueError("【処理停止】既存ロックの session_id が resume_state と一致しません。")

try:
    st_dt = datetime.fromisoformat(old_lock.get("start_time", ""))
    hb_dt = datetime.fromisoformat(old_lock.get("heartbeat", ""))
    if st_dt.tzinfo is None or hb_dt.tzinfo is None:
        raise ValueError("timezone-awareではありません")
except Exception as e:
    raise ValueError(f"【処理停止】既存ロックの日時形式が不正です: {e}")

if runtime_session_id != old_lock["session_id"]:
    raise RuntimeError(f"【処理停止】メモリ上の session_id ({runtime_session_id}) が既存ロックの session_id ({old_lock['session_id']}) と一致しません。\n自動引継ぎは行いません。")

print("✅ メモリ上のsession_idと既存ロックが一致しました。正当な所有者として移行を続行します。")

# 【2. 変更前のハッシュ】
def get_file_sha256(filepath):
    with open(filepath, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

manifest_sha_before = get_file_sha256(RUN_MANIFEST_PATH)
resume_sha_before = get_file_sha256(RESUME_STATE_PATH)
lock_sha_before = get_file_sha256(PROCESSING_LOCK_PATH)

# 【3. 新しいlock内容】
current_time = datetime.now(timezone.utc)
new_lock = dict(old_lock) # 未知のキーを保持

new_lock["config_hash"] = config_hash
new_lock["heartbeat"] = current_time.isoformat()
new_lock["status"] = "running"
new_lock["lock_schema_version"] = "1.0"
new_lock["migration"] = {
    "migrated_at": current_time.isoformat(),
    "previous_lock_sha256": lock_sha_before,
    "reason": "add config_hash and lock_schema_version to legacy initialization lock",
    "previous_keys": list(old_lock.keys())
}

# 【4. 既存JSONを更新する原子的関数】
print("\n【4. 原子的更新の実行】")
temp_lock_path = CHECKPOINT_ROOT / f".processing_lock_{runtime_session_id}.migration.tmp.json"

if temp_lock_path.exists():
    raise FileExistsError(f"【処理停止】一時ファイルが既に存在します: {temp_lock_path}")

with open(temp_lock_path, "x", encoding="utf-8") as f:
    json.dump(new_lock, f, indent=4, ensure_ascii=False)
    f.flush()
    os.fsync(f.fileno())

with open(temp_lock_path, "r", encoding="utf-8") as f:
    read_temp_lock = json.load(f)

if read_temp_lock != new_lock:
    raise RuntimeError("【処理停止】一時ファイルの内容がnew_lockと一致しません。")

# 直前ハッシュ確認
if get_file_sha256(PROCESSING_LOCK_PATH) != lock_sha_before:
    raise RuntimeError("【処理停止】処理中に既存ロックが変化しました。競合の可能性があります。")
if get_file_sha256(RUN_MANIFEST_PATH) != manifest_sha_before:
    raise RuntimeError("【処理停止】処理中にmanifestが変化しました。")
if get_file_sha256(RESUME_STATE_PATH) != resume_sha_before:
    raise RuntimeError("【処理停止】処理中にresume_stateが変化しました。")

os.replace(temp_lock_path, PROCESSING_LOCK_PATH)

with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
    final_read_lock = json.load(f)

if final_read_lock != new_lock:
    raise RuntimeError("【処理停止】置換後のロック内容がnew_lockと一致しません。")
if temp_lock_path.exists():
    raise RuntimeError("【処理停止】一時ファイルが削除されずに残っています。")

# 【5. 移行後の検証】
print("\n【5. 移行後の検証】")
lock_sha_after = get_file_sha256(PROCESSING_LOCK_PATH)
manifest_sha_after = get_file_sha256(RUN_MANIFEST_PATH)
resume_sha_after = get_file_sha256(RESUME_STATE_PATH)

validation_results = {
    "session_id_match": final_read_lock["session_id"] == old_lock["session_id"],
    "session_id_resume_match": final_read_lock["session_id"] == current_resume["current_session_id"],
    "config_hash_match": final_read_lock["config_hash"] == config_hash,
    "schema_version_match": final_read_lock["lock_schema_version"] == "1.0",
    "start_time_match": final_read_lock["start_time"] == old_lock["start_time"],
    "heartbeat_tz_aware": datetime.fromisoformat(final_read_lock["heartbeat"]).tzinfo is not None,
    "status_running": final_read_lock["status"] == "running",
    "migration_sha_match": final_read_lock["migration"]["previous_lock_sha256"] == lock_sha_before,
    "manifest_sha_unchanged": manifest_sha_after == manifest_sha_before,
    "resume_sha_unchanged": resume_sha_after == resume_sha_before,
    "lock_sha_updated": lock_sha_after != lock_sha_before,
    "batches_empty": len(list(BATCH_DIR.iterdir())) == 0,
    "errors_empty": len(list(ERROR_DIR.iterdir())) == 0,
    "temporary_empty": len(list(TEMP_DIR.iterdir())) == 0,
    "finals_not_exist": not FINAL_PICKLE_PATH.exists() and not FINAL_JSON_PATH.exists()
}

all_valid = all(validation_results.values())
for k, v in validation_results.items():
    if not v:
        print(f"  [失敗] {k}")

if not all_valid:
    raise RuntimeError("【処理停止】移行後の検証に失敗しました。")

# 【7. 最終表示】
added_keys = set(final_read_lock.keys()) - set(old_lock.keys())

print("\n--- 移行完了サマリー ---")
print(f"runtime_session_id: {runtime_session_id}")
print(f"lock.session_id: {final_read_lock['session_id']}")
print(f"config_hash: {final_read_lock['config_hash']}")
print(f"manifest SHA-256: {manifest_sha_before} -> {manifest_sha_after} (変更なし)")
print(f"resume_state SHA-256: {resume_sha_before} -> {resume_sha_after} (変更なし)")
print(f"processing_lock SHA-256: {lock_sha_before} -> {lock_sha_after} (更新)")
print(f"追加されたキー: {list(added_keys)}")
print(f"start_time: {final_read_lock['start_time']}")
print(f"更新後heartbeat: {final_read_lock['heartbeat']}")
print(f"lock_schema_version: {final_read_lock['lock_schema_version']}")
print(f"migration情報: {final_read_lock['migration']}")
print(f"バッチファイル数: batches={len(list(BATCH_DIR.iterdir()))}, errors={len(list(ERROR_DIR.iterdir()))}, temporary={len(list(TEMP_DIR.iterdir()))}")
print(f"\n移行判定: {'成功' if all_valid else '失敗'}")
if all_valid:
    print("\n✅ processing_lock.json のスキーマ移行が安全に完了しました。次に進むことができます。")

--- processing_lock.json の安全なスキーマ移行 ---

【1. ランタイムのロック所有権確認】


RuntimeError: 【処理停止】メモリ上の session_id (06be6212-53f4-4872-b7ac-ec56ad7d9b3b) が既存ロックの session_id (15a525f6-b3ce-4736-a8c2-3ea9db6cae49) と一致しません。
自動引継ぎは行いません。

In [11]:
import json
import uuid
from datetime import datetime, timezone

print("--- チェックポイント初期化とロック取得 ---")

# 過去のロックが残っている場合に強制解除するためのフラグ
# （ハートビートから30分以上経過している場合のみ有効です）
FORCE_CLEAR_LOCK = False

session_id = str(uuid.uuid4())
current_time = datetime.now(timezone.utc)

# 【1. ロックの確認】
if PROCESSING_LOCK_PATH.exists():
    with open(PROCESSING_LOCK_PATH, "r", encoding="utf-8") as f:
        try:
            existing_lock = json.load(f)
            last_heartbeat = datetime.fromisoformat(existing_lock["heartbeat"])
            time_diff = current_time - last_heartbeat

            if time_diff.total_seconds() < 30 * 60:
                raise RuntimeError(f"【処理停止】他のセッションが実行中の可能性があります。\n最後のheartbeat: {last_heartbeat}\n経過時間: {time_diff.total_seconds() / 60:.1f}分")
            else:
                if not FORCE_CLEAR_LOCK:
                    print(f"【確認】過去のランタイム切断等によりロックが残っている可能性があります。\n前回のheartbeat: {last_heartbeat}\n経過時間: {time_diff.total_seconds() / 60:.1f}分")
                    print("自動的に乗っ取ることはしません。安全を確認の上、再開する場合はこのセルの FORCE_CLEAR_LOCK = True に変更して再実行してください。")
                    raise RuntimeError("古い処理ロックが残っています。")
                else:
                    print("FORCE_CLEAR_LOCKがTrueに設定されているため、古いロックを解除して処理を引き継ぎます。")
        except json.JSONDecodeError:
            print("ロックファイルが破損しているため上書きします。")

# 【2. ディレクトリの作成】
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
BATCH_DIR.mkdir(exist_ok=True)
ERROR_DIR.mkdir(exist_ok=True)
TEMP_DIR.mkdir(exist_ok=True)

# 【3. マニフェストと進捗状態の保存（新規のみ）】
if mode == "新規":
    with open(RUN_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_preview, f, indent=4, ensure_ascii=False)

    initial_resume_state = {
        "pipeline_version": CONFIG["pipeline_version"],
        "config_hash": config_hash,
        "status": "initialized",
        "completed_batch_ids": [],
        "completed_source_files": [],
        "pending_batch_ids": list(batches_config.keys()),
        "failed_batch_ids": [],
        "total_batches": total_batches,
        "completed_batches": 0,
        "completed_files": 0,
        "total_files": total_files,
        "last_completed_batch": None,
        "last_completed_source_file": None,
        "last_update": current_time.isoformat(),
        "current_session_id": session_id
    }
    with open(RESUME_STATE_PATH, "w", encoding="utf-8") as f:
        json.dump(initial_resume_state, f, indent=4, ensure_ascii=False)

    print(f"新規チェックポイント環境を構築しました。")
    print(f"  - {RUN_MANIFEST_PATH.name}")
    print(f"  - {RESUME_STATE_PATH.name}")
else:
    print("既存のチェックポイント環境を引き継ぎます。")

# 【4. ロックの取得（上書き）】
lock_data = {
    "session_id": session_id,
    "start_time": current_time.isoformat(),
    "heartbeat": current_time.isoformat(),
    "status": "running"
}
with open(PROCESSING_LOCK_PATH, "w", encoding="utf-8") as f:
    json.dump(lock_data, f, indent=4, ensure_ascii=False)

print("\n✅ 処理ロックを取得しました。")
print(f"  session_id: {session_id}")


--- チェックポイント初期化とロック取得 ---


RuntimeError: 【処理停止】他のセッションが実行中の可能性があります。
最後のheartbeat: 2026-07-30 15:09:27.030972+00:00
経過時間: 29.6分